Cell 1: install the packages

In [0]:
%pip install -U -q databricks-mcp "databricks-sdk[openai]"
dbutils.library.restartPython()

Cell 2: run the test

In [0]:
from databricks.sdk import WorkspaceClient
from databricks_mcp import DatabricksMCPClient
from concurrent.futures import ThreadPoolExecutor

GENIE_SPACE_ID = "01f1ca11416719c5be1d8d5f9bd5619c"

w = WorkspaceClient()
url = f"{w.config.host.rstrip('/')}/api/2.0/mcp/genie/{GENIE_SPACE_ID}"
c = DatabricksMCPClient(server_url=url, workspace_client=w)
pool = ThreadPoolExecutor(1)

tools = pool.submit(c.list_tools).result()
print("Genie tools:", [t.name for t in tools])
try:
    r = pool.submit(c.call_tool, tools[0].name,
                    {"query": "Net sales by country for Q3 2026 vs Q3 2025"}).result()
    print("OK:", r.content)
except BaseException as e:
    print("ERROR:", repr(e))
    for sub in getattr(e, "exceptions", []):
        print("  ->", repr(sub))